# `forex(base, quote, start, end=None)`

Daily rates for any pair of stored currencies, from FMP via `forex.prices`. Change the arguments and re-run any cell.

> **About the data shown.** This function's data comes from FMP's API, and FMP's license doesn't allow displaying it publicly. So the outputs saved here are synthetic: made-up data from [`_synthetic.py`](_synthetic.py), run through the function's own, unchanged code. The values are not real prices or yields.
>
> To see real data, start Jupyter with `ARORA_REAL_DATA=1`: every cell then reads prod, read-only, as `readonly_user`. Don't commit those outputs; `tests/test_notebooks_offline.py` fails if you save them.

In [1]:
# Synthetic data in place of FMP's (see the note above), unless Jupyter was started with ARORA_REAL_DATA=1.
import _synthetic

_synthetic.use()

Data: synthetic


In [2]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.forex import FxBar
from arora_data.frames import to_df
from arora_data.sync import forex


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## A stored pair
FMP daily bars, returned as stored (`route` = stored).

In [3]:
show(forex("USD", "CAD", today_et() - timedelta(days=7)))

6 rows

pair|date|route|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---
USDCAD|2026-11-16|stored|1.070584|1.079638|1.068761|1.077570|6336|1.075323
USDCAD|2026-11-17|stored|1.078742|1.079430|1.073270|1.074397|4294|1.075699
USDCAD|2026-11-18|stored|1.073255|1.074883|1.070401|1.070772|2723|1.072019
USDCAD|2026-11-19|stored|1.070603|1.071818|1.063597|1.065832|7502|1.067082
USDCAD|2026-11-20|stored|1.063933|1.076052|1.060823|1.074706|1123|1.070527
USDCAD|2026-11-23|stored|1.073501|1.074615|1.067471|1.069222|3131|1.070436

## An inverted pair
Only USDCAD is stored, so CAD/USD is 1 / USDCAD. High and low swap. volume is FMP's tick count, the same either way round; vwap is 1 / vwap, a close approximation.

In [4]:
show(forex("CAD", "USD", today_et() - timedelta(days=7)))

6 rows

pair|date|route|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---
CADUSD|2026-11-16|inverted|0.934069629286|0.935662884405|0.926236386641|0.928013957330|6336|0.929953139661
CADUSD|2026-11-17|inverted|0.927005715917|0.931731996608|0.926414867106|0.930754646560|4294|0.929628083693
CADUSD|2026-11-18|inverted|0.931745018658|0.934229321535|0.930333813075|0.933905630704|2723|0.932819287718
CADUSD|2026-11-19|inverted|0.934053052345|0.940205735819|0.932994221034|0.938234168237|7502|0.937135103019
CADUSD|2026-11-20|inverted|0.939908810047|0.942664327602|0.929323118214|0.930487035524|1123|0.934119363641
CADUSD|2026-11-23|inverted|0.931531502998|0.936793599077|0.930565830553|0.935259469034|3131|0.934198775079

## A crossed pair
BRL/INR isn't stored, so it is usd_per(BRL) / usd_per(INR), from USDBRL and USDINR. A cross has open and close only, and skips a date either USD pair lacks.

In [5]:
show(forex("BRL", "INR", today_et() - timedelta(days=7)))

6 rows

pair|date|route|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---
BRLINR|2026-11-16|crossed|0.864023961233|None|None|0.868772649973|None|None
BRLINR|2026-11-17|crossed|0.869510171549|None|None|0.870751463413|None|None
BRLINR|2026-11-18|crossed|0.871528278857|None|None|0.878115113082|None|None
BRLINR|2026-11-19|crossed|0.878089322930|None|None|0.878373167379|None|None
BRLINR|2026-11-20|crossed|0.880226614055|None|None|0.875185360015|None|None
BRLINR|2026-11-23|crossed|0.877387619802|None|None|0.870463710157|None|None

## `forex("ALL", "ALL", start, end)`
Every pair of stored currencies once: stored pairs as stored, the rest crossed. Keep the range short: in prod, one week is about 50,000 rows.

In [6]:
to_df(forex("ALL", "ALL", today_et() - timedelta(days=3)), FxBar)

pair,date,route,open,high,low,close,volume,vwap
str,date,str,"decimal[38,12]","decimal[38,6]","decimal[38,6]","decimal[38,12]",i64,"decimal[38,6]"
"""AUDBRL""",2026-11-20,"""crossed""",1.161702591100,null,null,1.163737372140,null,null
"""AUDBRL""",2026-11-23,"""crossed""",1.162662099830,null,null,1.163562732300,null,null
"""AUDCAD""",2026-11-20,"""crossed""",1.181411417930,null,null,1.192670029380,null,null
"""AUDCAD""",2026-11-23,"""crossed""",1.191655887570,null,null,1.187413799880,null,null
"""AUDCHF""",2026-11-20,"""crossed""",1.179057329650,null,null,1.170983021300,null,null
…,…,…,…,…,…,…,…,…
"""MXNTWD""",2026-11-23,"""crossed""",0.981376290657,null,null,0.973555226815,null,null
"""USDMXN""",2026-11-20,"""stored""",1.020365000000,1.029212,1.017879,1.025483000000,4411,1.024191
"""USDMXN""",2026-11-23,"""stored""",1.024769000000,1.032435,1.023110,1.031546000000,6630,1.029030


## Full history in one call
EUR/USD from 1999, the start of FMP's history. There is no row limit, so the whole series comes back.

In [7]:
eurusd = to_df(forex("EUR", "USD", date(1999, 1, 1)), FxBar)
print(f"{eurusd.height:,} days, {eurusd['date'].min()} to {eurusd['date'].max()}")
eurusd.select("date", "open", "high", "low", "close").tail(5)

7,238 days, 1999-01-04 to 2026-11-23


date,open,high,low,close
date,"decimal[38,6]","decimal[38,6]","decimal[38,6]","decimal[38,6]"
2026-11-17,1.109292,1.112548,1.107570,1.109098
2026-11-18,1.110080,1.116321,1.106554,1.114388
2026-11-19,1.112429,1.115969,1.111892,1.115471
2026-11-20,1.115278,1.132558,1.107900,1.132055
2026-11-23,1.129110,1.146162,1.126559,1.140409


## Year-end closes
The last close of each year, from the frame above.

In [8]:
(
    eurusd.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

year,last_day,close
i32,date,"decimal[38,6]"
2017,2017-12-29,0.882358
2018,2018-12-31,0.840535
2019,2019-12-31,0.885949
2020,2020-12-31,0.902757
2021,2021-12-31,0.885338
2022,2022-12-30,0.919080
2023,2023-12-29,1.024286
2024,2024-12-31,1.019851
2025,2025-12-31,0.982236


## Several currencies in rupees, side by side
What one unit of each costs in INR. USD/INR is stored; the others are crossed through USD.

In [9]:
start = today_et() - timedelta(days=14)
rows = [r for base in ["USD", "EUR", "GBP", "JPY", "CNY"] for r in forex(base, "INR", start)]
to_df(rows, FxBar).pivot(on="pair", index="date", values="close")

date,USDINR,EURINR,GBPINR,JPYINR,CNYINR
date,"decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]","decimal[38,12]"
2026-11-09,0.926156000000,1.031739636310,1.021693622180,1.058898450900,0.967969372998
2026-11-10,0.919106000000,1.030716718000,1.020247181560,1.049347915190,0.957724317664
2026-11-11,0.917814000000,1.037112381530,1.020642209300,1.045248621430,0.953755315303
2026-11-12,0.910771000000,1.022293998180,1.020692862760,1.032162612720,0.947116599503
2026-11-13,0.910685000000,1.011203678240,1.026826479420,1.028721537990,0.939376372546
…,…,…,…,…,…
2026-11-17,0.911709000000,1.011174628480,1.019394596820,1.034034213410,0.947251781333
2026-11-18,0.917301000000,1.022229226790,1.023899631910,1.046225244420,0.952964752921
2026-11-19,0.920254000000,1.026516649630,1.026912358850,1.049951339060,0.964726947554


## An exotic cross
One Kuwaiti dinar in Lebanese pounds. Computed rates keep 12 significant digits, so large and tiny crosses both stay exact in a dataframe.

In [10]:
show(forex("KWD", "LBP", today_et() - timedelta(days=7)))

6 rows

pair|date|route|open|high|low|close|volume|vwap
---|---|---|---|---|---|---|---|---
KWDLBP|2026-11-16|crossed|0.861232723195|None|None|0.851750508665|None|None
KWDLBP|2026-11-17|crossed|0.851218234238|None|None|0.843971545445|None|None
KWDLBP|2026-11-18|crossed|0.842206004612|None|None|0.832102655837|None|None
KWDLBP|2026-11-19|crossed|0.832218193479|None|None|0.839191441553|None|None
KWDLBP|2026-11-20|crossed|0.839390085632|None|None|0.840515486636|None|None
KWDLBP|2026-11-23|crossed|0.840500929497|None|None|0.847833420249|None|None

## Bad input
A currency with no stored USD pair (gold XAU, IMF units, retired currencies), the same currency twice, or only one side "ALL" raise a clear error instead of returning nothing.

In [11]:
for args in [("XAU", "USD"), ("EUR", "EUR"), ("ALL", "USD")]:
    try:
        forex(*args, today_et() - timedelta(days=7))
    except ValueError as err:
        print(f"{args}: {err}")

('XAU', 'USD'): XAU has no stored USD pair, so XAU/USD can't be computed
('EUR', 'EUR'): base and quote must differ
('ALL', 'USD'): base and quote must both be "ALL" or both be currencies
